# Advanced RAG System

## 1. Setup & Configuration
Importing dependencies and setting up the environment.

In [15]:
import json
import sys
import os
from pathlib import Path
from datetime import datetime
from typing import List, Dict
from dotenv import load_dotenv
from sentence_transformers import CrossEncoder
from langchain_community.document_loaders import UnstructuredPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough,RunnableParallel
from langchain_chroma import Chroma
from langchain_weaviate import WeaviateVectorStore
import weaviate
import time
from rank_bm25 import BM25Okapi
import numpy as np

In [16]:
# Add parent directory to path and change to project root
# Get the notebook's current directory and find project root
notebook_dir = Path.cwd()
if notebook_dir.name == "notebooks":
    project_root = notebook_dir.parent
else:
    project_root = notebook_dir

# Change to project root and add to path
os.chdir(project_root)
sys.path.insert(0, str(project_root))

print(f"📂 Working directory: {os.getcwd()}")

from src.services.llm_services import (
    load_config,
    get_llm,
    get_text_embeddings,
    validate_api_keys,
    print_config_summary
)

from src.utils.cost_tracker import(get_accurate_cost)

# Load environment variables
load_dotenv()

# Load configuration from config.yaml
config = load_config("src/config/config.yaml")

# Validate API keys
validate_api_keys(config, verbose=True)

# Print summary
print_config_summary(config)

📂 Working directory: e:\my career life\AI Enginert Essential\Mini project 01
✅ Config loaded:
  LLM: openrouter (openai/gpt-4o-mini)
  Embeddings: sbert / sentence-transformers/all-MiniLM-L6-v2
  Temperature: 0.2
  Artifacts: ./artifacts


e:\my career life\AI Enginert Essential\Mini project 01\src\services\llm_services.py:375: UserWarning: ⚠️  COHERE_API_KEY not found in environment
  warnings.warn(f"⚠️  {key} not found in environment")


## 2. Model Initialization
Initializing the LLM, Embedding Model, and Cross-Encoder Reranker.

In [17]:
llm = get_llm(config)
embeddings = get_text_embeddings(config)

# Reranker (specific to this notebook - now from config)
rerank_cfg = config.get('rerank_settings', {})
reranker_model = rerank_cfg.get('model', "cross-encoder/ms-marco-MiniLM-L-6-v2")
reranker = CrossEncoder(reranker_model)

print(f"✅ LLM: {config['llm_provider']} / {config.get('openrouter_model', config.get('llm_model'))}")
print(f"✅ Embeddings: {config['text_emb_model']}")
print(f"✅ Reranker: {reranker_model}")

# Verify API key with test completion
print("\n🔍 Testing LLM API connection...")
try:
    test_response = llm.invoke("Say 'API working!' if you can read this.")
    test_msg = test_response.content if hasattr(test_response, 'content') else str(test_response)
    print(f"✅ LLM API verified: {test_msg[:50]}")
except Exception as e:
    print(f"❌ LLM API test failed: {e}")
    print("⚠️  Please check your .env file and API key configuration.")

✅ LLM: openrouter / gpt-4o-mini
✅ Embeddings: sentence-transformers/all-MiniLM-L6-v2
✅ Reranker: cross-encoder/ms-marco-MiniLM-L-6-v2

🔍 Testing LLM API connection...
✅ LLM API verified: API working!


## 3. Data Ingestion
Loading the saved chunks

In [18]:
chunks_json_path = config.get('chunks_json_path', 'artifacts/data_factory/chunks.json')

# Ensure directory exists
os.makedirs(os.path.dirname(chunks_json_path), exist_ok=True)

if os.path.exists(chunks_json_path):
    with open(chunks_json_path, 'r', encoding='utf-8') as f:
        chunks_data = json.load(f)
    
    # Safe keys to keep for Weaviate (now from config)
    vdb_cfg = config.get('vector_db', {})
    safe_keys = set(vdb_cfg.get('safe_keys', ['source', 'page_number', 'category', 'text_as_html']))
    
    # Convert dictionaries back to Document objects with cleaned metadata
    chunks = [
        Document(
            page_content=chunk['page_content'],
            metadata={k: v for k, v in chunk.get('metadata', {}).items() if k in safe_keys}
        )
        for chunk in chunks_data
    ]
    print(f"✅ Chunks loaded and cleaned: {len(chunks)} documents")
else:
    print(f"❌ {chunks_json_path} not found! Please run 01_data_factory.ipynb first.")

✅ Chunks loaded and cleaned: 682 documents


## 4. Dense Retrieval (Weaviate)
Building and querying the dense vector store using Weaviate.

In [19]:
client = weaviate.connect_to_local()

# 1. Reset collection to avoid schema conflicts (from config)
vdb_cfg = config.get('vector_db', {})
collection_name = vdb_cfg.get('collection_name', "AdvancedDense")
if client.collections.exists(collection_name):
    client.collections.delete(collection_name)
    print(f"🗑️ Existing collection '{collection_name}' deleted.")


SAFE_KEYS = set(vdb_cfg.get('safe_keys', ['source', 'page_number', 'category', 'text_as_html']))
for doc in chunks:
    doc.metadata = {k: v for k, v in doc.metadata.items() if k in SAFE_KEYS}

print("🔵 Building dense vector store with cleaned metadata...")

dense_vectorstore = WeaviateVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    client=client,
    index_name=collection_name,
    text_key="page_content"
)

print(f"✅ Dense index built: {len(chunks)} chunks")

🗑️ Existing collection 'AdvancedDense' deleted.
🔵 Building dense vector store with cleaned metadata...
✅ Dense index built: 682 chunks


e:\my career life\AI Enginert Essential\Mini project 01\.venv\Lib\site-packages\weaviate\warnings.py:302: ResourceWarning: Con004: The connection to Weaviate was not closed properly. This can lead to memory leaks.
            Please make sure to close the connection using `client.close()`.
  warnings.warn(
C:\Users\www\AppData\Local\Temp\ipykernel_11556\3806841388.py:17: ResourceWarning: unclosed <socket.socket fd=5488, family=23, type=1, proto=0, laddr=('::1', 5497, 0, 0), raddr=('::1', 8080, 0, 0)>
  dense_vectorstore = WeaviateVectorStore.from_documents(


In [20]:
def getting_dense_result(query:str, k: int = 10):
    dense_results = dense_vectorstore.similarity_search(query, k=k)
        
    return dense_results

## 5. Sparse Retrieval (BM25)
Building and querying the sparse index using BM25Okapi.

In [21]:
print("🟠 Building BM25 index...")

# Tokenize corpus (using chunks, not documents)
tokenized_corpus = [doc.page_content.lower().split() for doc in chunks]
bm25 = BM25Okapi(tokenized_corpus)

print(f"✅ BM25 index built: {len(chunks)} documents")

def bm25_search(query: str, top_k: int = 10):
    """
    Search using BM25 (sparse retrieval algorithm).
    """
    # 1. Tokenize the query
    tokenized_query = query.lower().split()
    
    # 2. Get BM25 scores for all documents
    scores = bm25.get_scores(tokenized_query)
    
    # 3. Find top-k indices using np.argsort()
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    # 4. Create results list
    results = []
    
    # 5. Loop through top_indices and append dictionaries
    for idx in top_indices:
        results.append({
            "doc": chunks[idx],
            "score": float(scores[idx]),
            "doc_id": int(idx)
        })
    
    return results

🟠 Building BM25 index...
✅ BM25 index built: 682 documents


## 6. Hybrid Fusion (RRF)
Combining dense and sparse results using Reciprocal Rank Fusion.

In [22]:
def rrf_fusion(dense_docs: List, bm25_results: List, k: int = 60) -> List:
    """
    Reciprocal Rank Fusion - combines dense and sparse retrieval.
    """
    # 1. Initialize an empty dictionary to store RRF scores
    rrf_scores = {}
    
    # Create a mapping to store document objects
    doc_map = {}

    # 2. Add dense scores
    for rank, doc in enumerate(dense_docs, 1):
        # Use page_content as unique key to prevent duplicates
        doc_key = doc.page_content
        score = 1.0 / (k + rank)
        rrf_scores[doc_key] = rrf_scores.get(doc_key, 0.0) + score
        if doc_key not in doc_map:
            doc_map[doc_key] = doc

    # 3. Add BM25 scores
    for rank, res in enumerate(bm25_results, 1):
        doc = res["doc"]
        # Use page_content as unique key to prevent duplicates
        doc_key = doc.page_content 
        score = 1.0 / (k + rank)
        rrf_scores[doc_key] = rrf_scores.get(doc_key, 0.0) + score
        if doc_key not in doc_map:
            doc_map[doc_key] = doc

    # 4. Sort by RRF score descending
    sorted_items = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

    # 5. Build final results list
    fused_docs = []
    for doc_key, score in sorted_items:
        fused_docs.append({
            "doc": doc_map[doc_key], 
            "score": score,
            "doc_id": doc_key  # using content as ID
        })

    return fused_docs

## 7. Reranking (Cross-Encoder)
Re-scoring top candidates for high precision.

In [23]:
def rerank(query: str, results: List, top_k: int = 3):
    """
    Rerank results using a cross-encoder for more accurate relevance scoring.
    """
    # 1. Prepare query-document pairs
    pairs = [[query, res["doc"].page_content] for res in results]
    
    # 2. Get scores from the cross-encoder
    scores = reranker.predict(pairs)
    
    # 3. Add rerank scores to results
    for i, res in enumerate(results):
        res["rerank_score"] = float(scores[i])
    
    # 4. Sort by rerank_score
    reranked = sorted(results, key=lambda x: x["rerank_score"], reverse=True)[:top_k]
    
    return reranked

## 8. LLM Response Generation

In [24]:
prompt_template = ChatPromptTemplate.from_template(
    """You are a Lead AI Architect and Senior Financial Analyst at Alpha-Yield Capital.
    Analyze the provided excerpts from Uber's Annual Report and answer the user's question.

    Context:
    {context}

    Question: {question}

    Instructions:
    1. Provide a direct, clear, and plain answer to the question based ONLY on the context.
    2. DO NOT mention phrases like "According to the document", "Based on the context", or "Section X".
    3. DO NOT cite sources or sections. Just provide the raw answer.
    4. If the answer is not in the context, state "Information not available".
    5. Maintain a professional and objective tone.

    Answer:"""
)

def llm_generation(context: str, query: str, llm):
    chain_with_context = (
        RunnableParallel(
            {
                "context": RunnablePassthrough(),
                "question": RunnablePassthrough(),
            }
        )
        | prompt_template
        | llm
        | StrOutputParser()
    )

    result = chain_with_context.invoke(
        {
            "context": context,
            "question": query,
        }
    )

    return result

## 9. Final RAG Pipeline
Orchestrating the complete Retrieval-Augmented Generation flow.

In [25]:
def format_context_with_tables(results: List) -> str:
    """
    Formats retrieval results into a string, prioritizing HTML tables for clarity.
    """
    formatted_chunks = []
    for res in results:
        doc = res["doc"]
        metadata = doc.metadata
        
        # Use HTML for tables if present
        if metadata.get('category') == 'Table' and metadata.get('text_as_html'):
            content = f"TABLE DATA (HTML):\n{metadata['text_as_html']}"
        else:
            content = doc.page_content
            
        formatted_chunks.append(f"--- SOURCE EXCERPT ---\n{content}")
        
    return "\n\n".join(formatted_chunks)

def query_librarian(
    query: str,
    dense_k: int = None,
    bm25_k: int = None,
    fusion_k: int = None,
    final_k: int = None,
    verbose: bool = False
) -> dict:
    
    # Set defaults from config
    rerank_cfg = config.get('rerank_settings', {})
    dense_k = dense_k or config.get('similarity_top_k', 10)
    bm25_k = bm25_k or config.get('similarity_top_k', 10)
    fusion_k = fusion_k or rerank_cfg.get('fusion_k', 6)
    final_k = final_k or rerank_cfg.get('final_k', 5)
    
    # 1. Dense Retrieval
    dense_results = getting_dense_result(query, k=dense_k)
    
    # 2. BM25 Retrieval
    bm25_results = bm25_search(query, top_k=bm25_k)
    
    # 3. RRF Fusion
    fused_results = rrf_fusion(dense_results, bm25_results)[:fusion_k]
    
    # 4. Reranking
    reranked_results = rerank(query, fused_results, top_k=final_k)

    # 5. Format Context (New: Table Aware)
    context_text = format_context_with_tables(reranked_results)
    
    # 6. LLM output
    answer = llm_generation(context=context_text, query=query, llm=llm)
    
    if verbose:
        print(f"\n🙋 Question: {query}")
        print(f"📚 Answer: {answer}")
        
    return {
        "answer": answer,
        "sources": reranked_results
    }

## 10. RAG Evaluation

In [26]:
test_data_path = config.get('golden_test_set_path', 'artifacts/test/golden_test_set.jsonl')
test_data=[]

# Ensure directory exists
os.makedirs(os.path.dirname(test_data_path ), exist_ok=True)

if os.path.exists(test_data_path ):
    with open(test_data_path , 'r', encoding='utf-8') as f:
        for line in f:
         test_data.append(json.loads(line))

In [27]:
results_list = []

print(f"🚀 Running evaluation for {len(test_data[:10])} questions...")

for i, data in enumerate(test_data[:10], 1):
    query = data['question']
    actual = data['answer']
    start_time = time.time()
    generated = query_librarian(query)
    elapsed = time.time() - start_time
    cost = get_accurate_cost(generated['sources'], query, generated['answer'], prompt_template)

    results_list.append({
        "id": i,
        "question": query,
        "actual_answer": actual,
        "generated_answer": generated,
        "latency":elapsed,
        "cost":cost
    })

🚀 Running evaluation for 10 questions...


In [28]:
import json

serializable_results = []

for item in results_list:
    serializable_results.append({
        "question": item['question'],
        "actual_answer": item['actual_answer'],
        "generated_answer": item['generated_answer']['answer'],
        "latency":item['latency'],
        "cost":item['cost']
    })

# Now save the cleaned list
output_path = config.get('eval_results_path', 'artifacts/test/rag_evaluation_results.json')
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(serializable_results, f, indent=4, ensure_ascii=False)

print(f"✅ Success! Saved only the answers to {output_path}")

✅ Success! Saved only the answers to ./artifacts/test/rag_evaluation_results.json
